In [0]:
df = spark.read.table("samples.bakehouse.sales_franchises")
display(df)

franchiseID,name,city,district,zipcode,country,size,longitude,latitude,supplierID
3000002,Golden Crumbs,San Francisco,Mission,94110,US,XL,-122.4194,37.7593,4000002
3000000,The Crumbly Nook,Sydney,Bondi Beach,2026,Australia,L,151.2763,-33.8915,4000000
3000001,Tokyo Treats,Tokyo,Shibuya,150-0042,Japan,M,139.701,35.6586,4000001
3000003,Sugar Rush,Melbourne,Fitzroy,3065,Australia,S,144.9804,-37.8005,4000003
3000004,Kobe Konfections,Kobe,Kitano,657-0838,Japan,XXL,135.2216,34.6806,4000004
3000005,Floured Fantasies,Los Angeles,Silver Lake,90026,US,XL,-118.2674,34.0877,4000005
3000006,Crumby Delights,Brisbane,West End,4101,Australia,M,153.0093,-27.4792,4000006
3000007,Kyoto Kravings,Kyoto,Gion,605-0811,Japan,S,135.9792,35.0036,4000007
3000008,Doughy Dreams,Chicago,Wicker Park,60622,US,XXL,-87.6782,41.9064,4000008
3000009,The Baking Lab,Perth,Leederville,6007,Australia,L,115.8344,-31.9393,4000009


In [0]:
df.printSchema()

root
 |-- franchiseID: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- country: string (nullable = true)
 |-- size: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- supplierID: long (nullable = true)



In [0]:
print(df.columns)

['franchiseID', 'name', 'city', 'district', 'zipcode', 'country', 'size', 'longitude', 'latitude', 'supplierID']


In [0]:
new_data = [(2000009, "Hololive", "Tokyo", "Saitama", "12345", "Japan", "Small", 139.7514, 35.6895, 2000001),
            (2000010, "TikTok", "Tokyo", "Saitama", "12345", "Japan", "Small", 139.7514, 35.6895, 2000001),
            (2000011, "Bing", "Tokyo", "Saitama", "12345", "Japan", "Small", 139.7514, 35.6895, 2000001)]

new_df = spark.createDataFrame(new_data, schema=df.schema)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("luffy.phase2.franchise")

In [0]:
new_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("luffy.phase2.franchise")

In [0]:
display(new_df)

franchiseID,name,city,district,zipcode,country,size,longitude,latitude,supplierID
2000009,Hololive,Tokyo,Saitama,12345,Japan,Small,139.7514,35.6895,2000001
2000010,TikTok,Tokyo,Saitama,12345,Japan,Small,139.7514,35.6895,2000001
2000011,Bing,Tokyo,Saitama,12345,Japan,Small,139.7514,35.6895,2000001


# Versioning

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "luffy.phase2.franchise")
history_df = delta_table.history()
display(history_df)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-03-02T13:49:32.000Z,76350293862913,this15anmoldesu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2949730483272454),353933c2-c110-4e60-bae8-0da38823dea2,0302-131908-hy3g79hp-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 2846)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-02T13:49:30.000Z,76350293862913,this15anmoldesu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2949730483272454),bf51ce8d-48eb-4b70-aa86-007a7d5abf9b,0302-131908-hy3g79hp-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 48, numOutputBytes -> 5245)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


In [0]:
df_old = spark.read.option("versionAsOf", 0).table("luffy.phase2.franchise")
display(df_old)

franchiseID,name,city,district,zipcode,country,size,longitude,latitude,supplierID
3000002,Golden Crumbs,San Francisco,Mission,94110,US,XL,-122.4194,37.7593,4000002
3000000,The Crumbly Nook,Sydney,Bondi Beach,2026,Australia,L,151.2763,-33.8915,4000000
3000001,Tokyo Treats,Tokyo,Shibuya,150-0042,Japan,M,139.701,35.6586,4000001
3000003,Sugar Rush,Melbourne,Fitzroy,3065,Australia,S,144.9804,-37.8005,4000003
3000004,Kobe Konfections,Kobe,Kitano,657-0838,Japan,XXL,135.2216,34.6806,4000004
3000005,Floured Fantasies,Los Angeles,Silver Lake,90026,US,XL,-118.2674,34.0877,4000005
3000006,Crumby Delights,Brisbane,West End,4101,Australia,M,153.0093,-27.4792,4000006
3000007,Kyoto Kravings,Kyoto,Gion,605-0811,Japan,S,135.9792,35.0036,4000007
3000008,Doughy Dreams,Chicago,Wicker Park,60622,US,XXL,-87.6782,41.9064,4000008
3000009,The Baking Lab,Perth,Leederville,6007,Australia,L,115.8344,-31.9393,4000009


In [0]:
new_df.count(), df_old.count()

(3, 48)

In [0]:
df2 = spark.table("luffy.phase2.franchise")

### Comparing the difference between older and newer version

In [0]:
diff_df = df2.join(df_old, on="franchiseID", how="left_anti")
display(diff_df)

franchiseID,name,city,district,zipcode,country,size,longitude,latitude,supplierID
2000009,Hololive,Tokyo,Saitama,12345,Japan,Small,139.7514,35.6895,2000001
2000010,TikTok,Tokyo,Saitama,12345,Japan,Small,139.7514,35.6895,2000001
2000011,Bing,Tokyo,Saitama,12345,Japan,Small,139.7514,35.6895,2000001


## Recovery

In [0]:
#Saving with only one row on purpose
df3 = df2.limit(1)

In [0]:
df3.write.format("delta").mode("overwrite").saveAsTable("luffy.phase2.franchise")

In [0]:
display(history_df)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-03-02T14:02:14.000Z,76350293862913,this15anmoldesu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2949730483272454),970b6da7-d816-46b8-bb38-73356b97b608,0302-131908-hy3g79hp-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 8091, numDeletionVectorsRemoved -> 0, numOutputRows -> 1, numOutputBytes -> 2572)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-02T13:49:32.000Z,76350293862913,this15anmoldesu@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2949730483272454),353933c2-c110-4e60-bae8-0da38823dea2,0302-131908-hy3g79hp-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 2846)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-02T13:49:30.000Z,76350293862913,this15anmoldesu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2949730483272454),bf51ce8d-48eb-4b70-aa86-007a7d5abf9b,0302-131908-hy3g79hp-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 48, numOutputBytes -> 5245)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


In [0]:
display(df3)

franchiseID,name,city,district,zipcode,country,size,longitude,latitude,supplierID
3000002,Golden Crumbs,San Francisco,Mission,94110,US,XL,-122.4194,37.7593,4000002


In [0]:
df_recovered_from_complete_loss = spark.read.option("versionAsOf", 1).table("luffy.phase2.franchise")
display(df_recovered_from_complete_loss)

franchiseID,name,city,district,zipcode,country,size,longitude,latitude,supplierID
3000002,Golden Crumbs,San Francisco,Mission,94110,US,XL,-122.4194,37.7593,4000002
3000000,The Crumbly Nook,Sydney,Bondi Beach,2026,Australia,L,151.2763,-33.8915,4000000
3000001,Tokyo Treats,Tokyo,Shibuya,150-0042,Japan,M,139.701,35.6586,4000001
3000003,Sugar Rush,Melbourne,Fitzroy,3065,Australia,S,144.9804,-37.8005,4000003
3000004,Kobe Konfections,Kobe,Kitano,657-0838,Japan,XXL,135.2216,34.6806,4000004
3000005,Floured Fantasies,Los Angeles,Silver Lake,90026,US,XL,-118.2674,34.0877,4000005
3000006,Crumby Delights,Brisbane,West End,4101,Australia,M,153.0093,-27.4792,4000006
3000007,Kyoto Kravings,Kyoto,Gion,605-0811,Japan,S,135.9792,35.0036,4000007
3000008,Doughy Dreams,Chicago,Wicker Park,60622,US,XXL,-87.6782,41.9064,4000008
3000009,The Baking Lab,Perth,Leederville,6007,Australia,L,115.8344,-31.9393,4000009
